In [ ]:
%pip install "librosa==1.0.0" --quiet

# Solution — Module M4-CNN: spectrogram CNN บน ESC-50

เฉลยของ `exercise.ipynb` — ตัวเลขอ้างอิงเป็น "ช่วงที่คาดหมาย" เพราะการเทรน
CNN บน GPU แต่ละรอบให้ค่าต่างกันเล็กน้อย ให้เทียบกับค่าที่คุณรันได้จริง

## 0. เตรียมข้อมูล + baseline + เทรน CNN (ให้มาแล้ว — รันผ่าน)

In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import librosa  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
from joblib import Parallel, delayed  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402
from torch.utils.data import DataLoader, TensorDataset  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
SR = 22050

In [ ]:
from pathlib import Path  # noqa: E402

import requests  # noqa: E402
import zipfile  # noqa: E402

ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip"


def ensure_esc50() -> Path:
    """ดาวน์โหลด/แตกไฟล์ ESC-50 ครั้งเดียว แล้วคืน path ของโฟลเดอร์."""
    on_kaggle = Path("/kaggle").exists()
    base = (
        Path("/kaggle/tmp") if on_kaggle
        else Path.home() / ".cache" / "esc50"
    )
    found = base / "ESC-50-master"
    if (found / "meta" / "esc50.csv").exists():
        print("ใช้ ESC-50 ที่มีอยู่แล้ว:", found)
        return found
    base.mkdir(parents=True, exist_ok=True)
    zpath = base / "esc50-master.zip"
    if not zpath.exists():
        print("ดาวน์โหลด ESC-50 (~600 MB) — ใช้เวลา 2–5 นาที")
        with requests.get(ESC50_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(base)
    print("แตกไฟล์แล้วที่:", found)
    return found


ESC50 = ensure_esc50()


In [ ]:
meta = pd.read_csv(ESC50 / "meta" / "esc50.csv")
id2cat = (
    meta.drop_duplicates("target")
    .sort_values("target")["category"]
    .tolist()
)
paths = [ESC50 / "audio" / f for f in meta["filename"]]
y_all = meta["target"].to_numpy()
folds = meta["fold"].to_numpy()
print("จำนวนคลิป:", len(meta))

In [ ]:
def mel_row(path) -> np.ndarray:
    """คลิปเดียว → log-mel spectrogram (128, 216) float32."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
    return librosa.power_to_db(S).astype(np.float32)


mels = np.asarray(Parallel(n_jobs=-1)(
    delayed(mel_row)(p) for p in paths
))
tr, te = folds != 5, folds == 5
Xtr = torch.from_numpy(mels[tr]).unsqueeze(1)
Xte = torch.from_numpy(mels[te]).unsqueeze(1)
ytr = torch.from_numpy(y_all[tr])
yte = torch.from_numpy(y_all[te])
mean, std = Xtr.mean(), Xtr.std()
Xtr_n = (Xtr - mean) / std
Xte_n = (Xte - mean) / std
print("train:", tuple(Xtr_n.shape), "| test (fold 5):",
      tuple(Xte_n.shape))

In [ ]:
def mfcc_row(path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 40 ตัวเลข (mean+std ของ MFCC 20 ตัว)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)])


X_mfcc = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_row)(p) for p in paths
))
rf = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=SEED
).fit(X_mfcc[tr], y_all[tr])
acc_rf5 = accuracy_score(y_all[te], rf.predict(X_mfcc[te]))
print(f"RF บน MFCC — accuracy (fold 5) = {acc_rf5:.4f}")

In [ ]:
class SmallCNN(nn.Module):
    """CNN จิ๋ว 3 ชั้น conv — อ่าน spectrogram (1, in_mels, in_frames)."""

    def __init__(self, in_mels=128, in_frames=216, n_classes=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(
                64 * (in_mels // 8) * (in_frames // 8), n_classes
            ),
        )

    def forward(self, x):
        return self.net(x)


def train_cnn(Xa, ya, epochs=30, bs=32, seed=0, log_every=5):
    """เทรน CNN ด้วย Adam บน GPU — คืนโมเดลที่เทรนแล้ว."""
    torch.manual_seed(seed)
    model = SmallCNN(
        in_mels=Xa.shape[2], in_frames=Xa.shape[3]
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(
        TensorDataset(Xa, ya), batch_size=bs, shuffle=True,
        generator=g,
    )
    for ep in range(epochs):
        model.train()
        run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item()
        if log_every and (ep + 1) % log_every == 0:
            print(f"  epoch {ep + 1}/{epochs}: "
                  f"loss={run / len(dl):.4f}")
    return model


def cnn_accuracy(model, Xa, ya):
    model.eval()
    with torch.no_grad():
        pred = model(Xa.to(DEVICE)).argmax(1).cpu().numpy()
    return accuracy_score(ya.numpy(), pred)

เทรน CNN 30 epochs:

In [ ]:
cnn = train_cnn(Xtr_n, ytr, epochs=30)
acc_cnn = cnn_accuracy(cnn, Xte_n, yte)
print(f"CNN 30 epochs — accuracy (fold 5) = {acc_cnn:.4f}")

## ข้อ 1 — 60 epochs

In [ ]:
cnn60 = train_cnn(Xtr_n, ytr, epochs=60, log_every=10)
acc60 = cnn_accuracy(cnn60, Xte_n, yte)
print(f"CNN 60 epochs — accuracy (fold 5) = {acc60:.4f}")

**คำตอบข้อ 1:** โดยทั่วไป 60 epochs ให้ accuracy ใกล้เคียงหรือดีขึ้น
เพียง ~1–3 จุดจาก 30 epochs (ช่วงคาดหมาย ~0.65–0.78) — สังเกต loss ของ
ช่วงหลัง: train loss ลดต่อไปได้ แต่ถ้า accuracy บน fold 5 ไม่ขยับ นั่นคือ
สัญญาณ **overfit** (จำ train เก่งขึ้น แต่ไม่ทำให้ทำนายข้อมูลใหม่ดีขึ้น) —
บทเรียนเดียวกับข้อ 2 ของ M1: เทรนนานขึ้น ≠ ดีขึ้นเสมอ

## ข้อ 2 — n_mels=64

In [ ]:
def mel_row64(path) -> np.ndarray:
    """คลิปเดียว → log-mel spectrogram (64, 216) float32."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=64)
    return librosa.power_to_db(S).astype(np.float32)


mels64 = np.asarray(Parallel(n_jobs=-1)(
    delayed(mel_row64)(p) for p in paths
))
X64 = torch.from_numpy(mels64).unsqueeze(1)
mean64, std64 = X64[tr].mean(), X64[tr].std()
Xtr64 = (X64[tr] - mean64) / std64
Xte64 = (X64[te] - mean64) / std64
cnn64 = train_cnn(Xtr64, ytr, epochs=30)
acc64 = cnn_accuracy(cnn64, Xte64, yte)
print(f"CNN n_mels=64 — accuracy (fold 5) = {acc64:.4f}")

**คำตอบข้อ 2:** accuracy มักลดลงเล็กน้อย (ราว 1–5 จุดจาก n_mels=128 —
ช่วงคาดหมาย ~0.60–0.73) เพราะแถบ mel หยาบลง แยกเสียงที่ต่างกันแค่
ความถี่ละเอียด ๆ ได้น้อยลง แต่ภาพเล็กลงครึ่งหนึ่ง = เทรนเร็วขึ้นและใช้ RAM
น้อยลง สำหรับโจทย์จริง trade-off นี้คือ "ความแม่น vs ต้นทุน" ที่ต้องวัดจริง
— เหมือนการเลือกความละเอียดการสุ่มตัวอย่างในงานวัด

## ข้อ 3 — คู่สับสนของ CNN

In [ ]:
from collections import Counter  # noqa: E402

pred = cnn(Xte_n.to(DEVICE)).argmax(1).cpu().numpy()

wrong = Counter(
    (id2cat[t], id2cat[p])
    for t, p in zip(y_all[te], pred) if t != p
)
print(wrong.most_common(8))

**คำตอบข้อ 3:** คู่สับสนของ CNN มักเป็นเสียง "ร้อง/กระแทก" ใกล้กัน
เชิงเสียงเช่นเดียวกับ RF (เช่น สัตว์ร้องสองชนิด, เสียงเคาะกับเสียงหยดน้ำ)
แต่โดยรวมจำนวนคู่สับสนของ CNN น้อยกว่า RF ชัดเจน (accuracy สูงกว่า) —
สิ่งที่ควรสังเกต: คู่ที่ RF พลาดหนัก (เช่น `breathing → keyboard_typing`)
มักถูก CNN แก้ได้ เพราะ CNN เห็น **ลวดลายตามเวลา** ทั้ง spectrogram ไม่ใช่
แค่ค่าเฉลี่ย ทิ้งช่วง — นี่คือคำตอบว่าข้อมูลส่วนไหนที่ MFCC 40 ตัวเลขทิ้งไป

## ข้อ 4 — ตาราง bake-off

**ตารางตัวอย่างจากรอบที่รัน (ค่าของคุณอาจต่างเล็กน้อย — ใช้ของคุณเอง):**

| วิธี | input | accuracy (fold 5) |
|---|---|---|
| RF บน MFCC | ตาราง 40 ตัวเลข | ~0.45–0.50 |
| CNN 30 epochs (n_mels=128) | ภาพ 128×216 | ~0.65–0.75 |
| CNN 60 epochs (ข้อ 1) | ภาพ 128×216 | ~0.65–0.78 |
| CNN 30 epochs (n_mels=64, ข้อ 2) | ภาพ 64×216 | ~0.60–0.73 |

**คำตอบข้อ 4:** ก้าวกระโดดจาก RF → CNN มาจาก "ข้อมูลที่โมเดลเห็น": MFCC
40 ตัวเลขคือผลของการบีบด้วยสมมติฐานของมนุษย์ (mel + DCT + สรุป mean/std
ทิ้งลวดลายตามเวลา) ส่วน CNN อ่าน spectrogram เต็มและเรียนรู้เองว่า pattern
ไหนสำคัญ — เท่ากับ M1 ที่ CNN ชนะ HOG+SVM เพราะเหตุผลเดียวกัน

## ข้อ 5 — เครื่องจักร 200 คลิป ไม่มี GPU

**คำตอบข้อ 5:** เลือก **RF บน MFCC** ก่อน:
1. **200 คลิปน้อยเกินสำหรับ CNN จากศูนย์** — โมเดลที่ต้องเรียนรู้ feature
   เองต้องการข้อมูลมาก (จำจาก learning curve ของ M1: ข้อมูล 5% แล้ว
   CNN จากศูนย์จม) ที่ 200 ตัวอย่าง CNN มัก overfit ก่อนมีประโยชน์
2. **เครื่องปลายทางไม่มี GPU** — RF วิ่งบน CPU ได้ทันทีและต่อคลิปใช้
   มิลลิวินาที ส่วน CNN ต้อง deploy บน GPU/quantize ซึ่งเพิ่มต้นทุนโครงการ
   จริงมาก
3. **ตรวจสอบได้** — RF บอกได้ว่าโมเดลดู feature ไหน (สำคัญมากเมื่อ
   ผลต้องนำไปอธิบายวิศวกร/ผู้กำกับคุณภาพ) ขณะที่ CNN เป็นกล่องดำ
   (ตรวจได้เฉพาะบางส่วนด้วยเทคนิคแบบ Grad-CAM — M7)

ถ้าวัดแล้ว RF ยังไม่พอ ทางขั้นกลางที่ควรลองก่อน CNN: เพิ่ม feature
(delta-MFCC — ข้อ 1 ของ exercise ครึ่งแรก), สแต็กโมเดลตารางหลายตัว
แล้วค่อยพิจารณา CNN พร้อมข้อมูลเพิ่ม — และตรวจ license ของข้อมูลที่จะเพิ่ม
(ESC-50 แจกต่อเชิงพาณิชย์ไม่ได้)

## สรุป (เฉลย)

- 60 epochs: accuracy ~ เดิมหรือ +1–3 จุด — หลักฐาน overfit อ่านจาก loss
- n_mels 64: แลกความละเอียดความถี่กับความเร็ว — ลดลงเล็กน้อย
- CNN ชนะ RF เพราะอ่านข้อมูลเต็มแทนที่จะกิน "สรุป 40 ตัวเลข"
- โจทย์จริง: ข้อมูลน้อย + ไม่มี GPU + ต้องอธิบายได้ → เริ่ม RF เสมอ